In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
seiftarek158_contract_nli_path = kagglehub.dataset_download('seiftarek158/contract-nli')

print('Data source import complete.')

100%|██████████| 29.1M/29.1M [00:00<00:00, 159MB/s]

Extracting files...


Data source import complete.


In [4]:
import os
import zipfile

# Your FIRST cell — just replace the extract_dir line:
extract_dir = seiftarek158_contract_nli_path
print(f"\nContents of {extract_dir}:")
for root, dirs, files in os.walk(extract_dir):
    level = root.replace(extract_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}\U0001f4c1 {os.path.basename(root)}/")
    for f in files:
        fpath = os.path.join(root, f)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"{indent}  \U0001f4c4 {f}  ({size_kb:.1f} KB)")


Contents of /root/.cache/kagglehub/datasets/seiftarek158/contract-nli/versions/1:
📁 1/
  📁 contract-nli/
    📄 test.json  (2207.2 KB)
    📄 dev.json  (1155.4 KB)
    📄 LICENSE  (18.2 KB)
    📄 train.json  (7429.9 KB)
    📄 README.md  (5.4 KB)
    📄 TERMS  (3.6 KB)
    📁 raw/
      📄 934545_0000891618-99-004640_document_2.txt  (26.5 KB)
      📄 1073090_0001356564-06-000012_sorell10ksbamend2x102.txt  (20.4 KB)
      📄 Non-Disclosure-NDA-UW-Oshkosh_FINALV2.pdf  (160.1 KB)
      📄 annex-iii---nda-agreement..pdf  (150.5 KB)
      📄 1693664_0001193125-18-171470_d426098dex99d3.htm  (37.3 KB)
      📄 Non_disclodure_contract.pdf  (34.5 KB)
      📄 Kenway-NDA-Form-Blank.pdf  (220.6 KB)
      📄 1053949_0001005150-98-000126_document_8.txt  (5.1 KB)
      📄 1023734_0000912057-96-023266_document_16.txt  (1.9 KB)
      📄 917639_0000912057-01-537118_a2062042zex-99_d7.htm  (28.5 KB)
      📄 1094814_0001140361-18-017998_s002178x1_ex99d7.htm  (64.6 KB)
      📄 sample-nrel-bilateral-nda-template.pdf  (63

In [10]:
import json

DATA_DIR = seiftarek158_contract_nli_path

with open(f"{DATA_DIR}/contract-nli/train.json") as f:
    train_data = json.load(f)

print("Top-level keys:", list(train_data.keys()))
print("Number of documents:", len(train_data["documents"]))
print("Number of hypotheses:", len(train_data["labels"]))
print("\nHypothesis IDs and texts:")
for h_id, h_text in train_data["labels"].items():
    print(f"  {h_id}: {h_text}")

Top-level keys: ['documents', 'labels']
Number of documents: 423
Number of hypotheses: 17

Hypothesis IDs and texts:
  nda-11: {'short_description': 'No reverse engineering', 'hypothesis': "Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information."}
  nda-16: {'short_description': 'Return of confidential information', 'hypothesis': 'Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.'}
  nda-15: {'short_description': 'No licensing', 'hypothesis': 'Agreement shall not grant Receiving Party any right to Confidential Information.'}
  nda-10: {'short_description': 'Confidentiality of Agreement', 'hypothesis': 'Receiving Party shall not disclose the fact that Agreement was agreed or negotiated.'}
  nda-2: {'short_description': 'None-inclusion of non-technical information', 'hypothesis': 'Confidential Information shall only include technical information.'}
  nda-1: {'short_description'

In [11]:
doc = train_data["documents"][0]

print("Document ID:", doc["id"])
print("Contract length (chars):", len(doc["text"]))
print("Number of spans:", len(doc["spans"]))
print("\nFirst 3 spans (char ranges):")
for i, span in enumerate(doc["spans"][:3]):
    print(f"  span[{i}]: chars {span[0]}\u2013{span[1]} \u2192 {repr(doc['text'][span[0]:span[1]][:80])}")

print("\nAnnotation keys:", list(doc["annotation_sets"][0]["annotations"].keys()))

print("\nSample annotation (nda-1):")
ann = doc["annotation_sets"][0]["annotations"]["nda-1"]
print("  choice:", ann["choice"])
print("  span indices:", ann["spans"])
if ann["spans"]:
    idx = ann["spans"][0]
    cs, ce = doc["spans"][idx]
    print(f"  resolved quote (span[{idx}], {cs}\u2013{ce}): {repr(doc['text'][cs:ce][:120])}")

Document ID: 34
Contract length (chars): 8585
Number of spans: 65

First 3 spans (char ranges):
  span[0]: chars 0–44 → 'NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT'
  span[1]: chars 45–132 → 'This NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT (“Agreement”) is made by and b'
  span[2]: chars 133–331 → '(i) the Office of the United Nations High Commissioner for Refugees, having its '

Annotation keys: ['nda-11', 'nda-16', 'nda-15', 'nda-10', 'nda-2', 'nda-1', 'nda-19', 'nda-12', 'nda-20', 'nda-3', 'nda-18', 'nda-7', 'nda-17', 'nda-8', 'nda-13', 'nda-5', 'nda-4']

Sample annotation (nda-1):
  choice: Entailment
  span indices: [14]
  resolved quote (span[14], 1294–1683): '1. “Confidential Information”, whenever used in this Agreement, shall mean any data, document, specification and other i'


## Section 0 — Setup & Config

All tunable constants live in Cell 0b only.  
Create a `.env` file in the project root with `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` before running Cell 0c.

In [6]:
# Cell 0a — Install (run once)
%pip install neo4j sentence-transformers python-dotenv peft transformers accelerate bitsandbytes pyyaml -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 7.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00:00:0100:01


In [30]:
from google.colab import userdata


In [21]:
# Cell 0b — Config (all tunable constants live here, nowhere else)
import os
from dotenv import load_dotenv, dotenv_values,find_dotenv
load_dotenv(override=True)  # loads vars into os.environ, returns bool
print(find_dotenv())
NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

# Embedding model
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM   = 384  # output dim of all-MiniLM-L6-v2

# Retrieval limits — change here, propagates everywhere
CHUNK_WINDOW  = 70    # approximate token window per eval chunk (fallback chunker)
CHUNK_STRIDE  = 35    # stride (50% overlap)
ANCHOR_TOP_K  = 3     # eval chunks used as query anchors per hypothesis
PER_LABEL_K   = 5     # precedents per label pulled from Neo4j
MAX_FEW_SHOTS = 5     # max examples shown in prompt (<=  PER_LABEL_K)

# Data path (set from kaggle download cell above)
DATASET_DIR = f"{seiftarek158_contract_nli_path}/contract-nli"

# Model paths (used in later sections, not in scope for Tasks 1-10)
ADAPTER_PATH   = "../results/models/qwen3-4B-nli-lora-adapter"
TEMPERATURE    = 0.3
MAX_NEW_TOKENS = 256

print(f"Neo4j      : {NEO4J_URI}")
print(f"Embed model: {EMBED_MODEL}, dim={EMBED_DIM}")
print(f"Anchor k   : {ANCHOR_TOP_K}, per-label k: {PER_LABEL_K}")
print(f"Dataset    : {DATASET_DIR}")

/content/.env
Neo4j      : neo4j+s://1070987b.databases.neo4j.io
Embed model: sentence-transformers/all-MiniLM-L6-v2, dim=384
Anchor k   : 3, per-label k: 5
Dataset    : /root/.cache/kagglehub/datasets/seiftarek158/contract-nli/versions/1/contract-nli


In [14]:
# Cell 0c — Neo4j driver + connection check
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as s:
    result = s.run("RETURN 1 AS ping").single()
    assert result["ping"] == 1, "Neo4j connection failed"
print("Neo4j connection OK")

Neo4j connection OK


In [12]:
# Cell 0d — Load ContractNLI splits
# Train split enters Neo4j; eval (dev) split stays in memory only.
import json

with open(f"{DATASET_DIR}/train.json") as f:
    _train = json.load(f)
with open(f"{DATASET_DIR}/dev.json") as f:
    _dev = json.load(f)

TRAIN_CONTRACTS = _train["documents"]
EVAL_CONTRACTS  = _dev["documents"]

print(f"Train contracts : {len(TRAIN_CONTRACTS)}")
print(f"Eval contracts  : {len(EVAL_CONTRACTS)}")
assert len(TRAIN_CONTRACTS) > 0
assert len(EVAL_CONTRACTS)  > 0

# Confirm span structure: ann["spans"] are indices into doc["spans"]
_d  = TRAIN_CONTRACTS[0]
_a  = _d["annotation_sets"][0]["annotations"]["nda-1"]
print(f"\nSpan structure confirmed:")
print(f"  doc['spans'] length    : {len(_d['spans'])}  (char ranges)")
print(f"  ann['spans'] for nda-1 : {_a['spans'][:3]}  (indices into doc['spans'])")

Train contracts : 423
Eval contracts  : 61

Span structure confirmed:
  doc['spans'] length    : 65  (char ranges)
  ann['spans'] for nda-1 : [14]  (indices into doc['spans'])


## Section 1 — Schema Creation (run once)

Creates constraints and the vector index in Neo4j. All queries use `IF NOT EXISTS` — safe to re-run.

In [34]:
# Cell 1a — Schema (idempotent)
SCHEMA_QUERIES = [
    "CREATE CONSTRAINT hyp_id IF NOT EXISTS FOR (h:Hypothesis) REQUIRE h.h_id IS UNIQUE",
    "CREATE CONSTRAINT contract_id IF NOT EXISTS FOR (c:Contract) REQUIRE c.contract_id IS UNIQUE",
    "CREATE CONSTRAINT clause_id IF NOT EXISTS FOR (cl:Clause) REQUIRE cl.clause_id IS UNIQUE",
    "CREATE CONSTRAINT ann_id IF NOT EXISTS FOR (a:Annotation) REQUIRE a.annotation_id IS UNIQUE",
    f"""CREATE VECTOR INDEX clause_embedding IF NOT EXISTS
        FOR (cl:Clause) ON cl.embedding
        OPTIONS {{indexConfig: {{`vector.dimensions`: {EMBED_DIM}, `vector.similarity_function`: 'cosine'}}}}""",
]

with driver.session() as s:
    for q in SCHEMA_QUERIES:
        s.run(q)
print("Schema created")

Schema created


In [36]:
# Cell 1b — Ingest Hypothesis nodes from playbook.yaml
import yaml

with open("./playbook.yaml") as f:
    playbook = yaml.safe_load(f)

CREATE_HYP = """
MERGE (h:Hypothesis {h_id: $h_id})
SET h.title       = $title,
    h.definition  = $definition,
    h.criticality = $criticality
"""

with driver.session() as s:
    for check in playbook["checks"]:
        s.run(CREATE_HYP, {
            "h_id":        check["hypothesis_id"],
            "title":       check["title"],
            "definition":  check["hypothesis_text"],
            "criticality": check["criticality"],
        })

# Assertion
with driver.session() as s:
    count = s.run("MATCH (h:Hypothesis) RETURN count(h) AS n").single()["n"]
    assert count == 17, f"Expected 17 Hypothesis nodes, got {count}"
print(f"Hypothesis nodes: {count}")

Hypothesis nodes: 17


## Section 2 — Corpus Ingestion (run once)

Embeds all annotated spans from the 423 training contracts and stores them in Neo4j.

**Span structure reminder:** `ann["spans"]` contains integer *indices* into `doc["spans"]` (the document-level list of `[char_start, char_end]` pairs).  
Clauses are batch-embedded per contract (one `encode()` call per contract, not per clause).

In [37]:
# Cell 2a — Load embedding model
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer(EMBED_MODEL)
print(f"Embedding model loaded: {EMBED_MODEL}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [38]:
# Cell 2b — Ingest training contracts (safe to re-run; uses MERGE)

INGEST_CONTRACT = "MERGE (c:Contract {contract_id: $cid})"

# UNWIND batches all clauses / annotations per contract → 2 queries per contract
# instead of 2*N queries per contract
INGEST_CLAUSES_BATCH = """
UNWIND $clauses AS row
MERGE (cl:Clause {clause_id: row.clause_id})
SET cl.text      = row.text,
    cl.embedding = row.embedding
WITH cl, row
MATCH (c:Contract {contract_id: row.cid})
MERGE (c)-[:HAS_CLAUSE]->(cl)
"""

INGEST_ANNOTATIONS_BATCH = """
UNWIND $annotations AS row
MERGE (a:Annotation {annotation_id: row.ann_id})
SET a.label = row.label
WITH a, row
MATCH (cl:Clause    {clause_id:   row.clause_id})
MATCH (h:Hypothesis {h_id:        row.h_id})
MATCH (c:Contract   {contract_id: row.cid})
MERGE (a)-[:SUPPORTED_BY]->(cl)
MERGE (a)-[:GROUNDS_HYPOTHESIS]->(h)
MERGE (a)-[:FOUND_IN]->(c)
"""

# Dataset uses "Entailment" / "Contradiction" (not "Entailed" / "Contradicted")
LABEL_MAP = {
    "Entailment":   "ENTAILED",
    "Contradiction": "CONTRADICTED",
    "NotMentioned": "NOT_MENTIONED",
}

# Dataset nda-keys are non-contiguous (skips nda-6, nda-9, nda-14)
# This maps each dataset key to the playbook's sequential H01-H17 IDs
NDA_TO_HID = {
    "nda-1":  "H01", "nda-2":  "H02", "nda-3":  "H03",
    "nda-4":  "H04", "nda-5":  "H05", "nda-7":  "H06",
    "nda-8":  "H07", "nda-10": "H08", "nda-11": "H09",
    "nda-12": "H10", "nda-13": "H11", "nda-15": "H12",
    "nda-16": "H13", "nda-17": "H14", "nda-18": "H15",
    "nda-19": "H16", "nda-20": "H17",
}


def ingest_contract(session, contract: dict):
    cid       = str(contract["id"])
    text      = contract["text"]
    doc_spans = contract["spans"]

    session.run(INGEST_CONTRACT, cid=cid)

    ann_set = contract["annotation_sets"][0]

    texts    = []
    metadata = []

    for h_key, ann in ann_set["annotations"].items():
        h_id  = NDA_TO_HID.get(h_key)
        if h_id is None:
            continue  # skip any key not in our playbook mapping
        label = LABEL_MAP.get(ann["choice"], "NOT_MENTIONED")
        for span_idx in ann.get("spans", []):
            char_start, char_end = doc_spans[span_idx]
            texts.append(text[char_start:char_end])
            metadata.append((
                f"{cid}_span_{span_idx}",
                f"ann_{cid}_{h_id}_{span_idx}",
                h_id, label,
            ))

    if not texts:
        return

    embeddings = embedder.encode(texts, batch_size=32, show_progress_bar=False)

    clauses     = []
    annotations = []
    for (clause_id, ann_id, h_id, label), clause_text, emb in zip(metadata, texts, embeddings):
        clauses.append({
            "clause_id": clause_id,
            "text":      clause_text,
            "embedding": emb.tolist(),
            "cid":       cid,
        })
        annotations.append({
            "ann_id":    ann_id,
            "label":     label,
            "clause_id": clause_id,
            "h_id":      h_id,
            "cid":       cid,
        })

    session.run(INGEST_CLAUSES_BATCH,     clauses=clauses)
    session.run(INGEST_ANNOTATIONS_BATCH, annotations=annotations)


with driver.session() as s:
    for i, contract in enumerate(TRAIN_CONTRACTS):
        ingest_contract(s, contract)
        if (i + 1) % 50 == 0:
            print(f"  Ingested {i+1}/{len(TRAIN_CONTRACTS)} contracts...")

print("Ingestion complete")

  Ingested 50/423 contracts...
  Ingested 100/423 contracts...
  Ingested 150/423 contracts...
  Ingested 200/423 contracts...
  Ingested 250/423 contracts...
  Ingested 300/423 contracts...
  Ingested 350/423 contracts...
  Ingested 400/423 contracts...
Ingestion complete


In [39]:
# Cell 2c — Verify ingestion
with driver.session() as s:
    n_contracts = s.run("MATCH (c:Contract)   RETURN count(c)  AS n").single()["n"]
    n_clauses   = s.run("MATCH (cl:Clause)    RETURN count(cl) AS n").single()["n"]
    n_anns      = s.run("MATCH (a:Annotation) RETURN count(a)  AS n").single()["n"]

print(f"Contracts  : {n_contracts}")
print(f"Clauses    : {n_clauses}")
print(f"Annotations: {n_anns}")
assert n_contracts == len(TRAIN_CONTRACTS), "Contract count mismatch"
assert n_clauses   > 0, "No clauses stored"

with driver.session() as s:
    null_emb = s.run(
        "MATCH (cl:Clause) WHERE cl.embedding IS NULL RETURN count(cl) AS n"
    ).single()["n"]
    assert null_emb == 0, f"{null_emb} clauses missing embeddings"

print("All clauses have embeddings — ingestion verified")

Contracts  : 423
Clauses    : 6139
Annotations: 8341
All clauses have embeddings — ingestion verified


## Section 3 — Hypothesis Embeddings (run once)

Embeds all 17 hypothesis definitions, stores them in Neo4j, and builds the in-memory `HYPOTHESES` dict used throughout retrieval.

In [40]:
# Cell 3a — Embed hypotheses + store in Neo4j + build in-memory dict
SET_HYP_EMB = "MATCH (h:Hypothesis {h_id: $h_id}) SET h.embedding = $embedding"

HYPOTHESES = {}  # h_id -> {title, definition, embedding (np.ndarray), criticality}

with driver.session() as s:
    for check in playbook["checks"]:
        h_id       = check["hypothesis_id"]
        definition = check["hypothesis_text"]
        embedding  = embedder.encode(definition).tolist()
        s.run(SET_HYP_EMB, h_id=h_id, embedding=embedding)
        HYPOTHESES[h_id] = {
            "title":       check["title"],
            "definition":  definition,
            "embedding":   np.array(embedding),
            "criticality": check["criticality"],
        }

assert len(HYPOTHESES) == 17

with driver.session() as s:
    no_emb = s.run(
        "MATCH (h:Hypothesis) WHERE h.embedding IS NULL RETURN count(h) AS n"
    ).single()["n"]
    assert no_emb == 0, f"{no_emb} hypotheses missing embeddings"

print(f"Embedded {len(HYPOTHESES)} hypotheses — all stored in Neo4j")

Embedded 17 hypotheses — all stored in Neo4j


## Section 4 — Eval Contract Chunker

Pure Python — nothing goes to Neo4j.  
Uses `contract["spans"]` (the document-level `[start, end]` pairs) if present; falls back to overlapping word-window sliding otherwise.

In [41]:
# Cell 4 — Eval contract chunker

def get_eval_chunks(
    contract: dict,
    window: int = CHUNK_WINDOW,
    stride: int = CHUNK_STRIDE,
) -> list:
    text  = contract["text"]
    spans = contract.get("spans", [])  # list of [char_start, char_end] pairs

    if spans:
        # Use pre-labeled spans directly (same granularity as training Clause nodes)
        return [
            {
                "chunk_id":   f"span_{idx}",
                "text":       text[span[0]:span[1]],
                "char_start": span[0],
                "char_end":   span[1],
            }
            for idx, span in enumerate(spans)
        ]

    # Fallback: overlapping word-window sliding
    words     = text.split()
    chunks    = []
    start_w   = 0
    chunk_idx = 0
    while start_w < len(words):
        end_w       = min(start_w + window, len(words))
        chunk_words = words[start_w:end_w]
        chunk_text  = " ".join(chunk_words)
        char_start  = text.find(chunk_words[0]) if chunk_words else 0
        chunks.append({
            "chunk_id":   f"window_{chunk_idx}",
            "text":       chunk_text,
            "char_start": char_start,
            "char_end":   char_start + len(chunk_text),
        })
        start_w   += stride
        chunk_idx += 1
        if end_w == len(words):
            break
    return chunks


# Assertion A: real eval contract (spans path if available)
_c0        = EVAL_CONTRACTS[0]
_chunks_s  = get_eval_chunks(_c0)
used_spans = bool(_c0.get("spans"))
print(f"Contract has top-level spans: {used_spans}")
print(f"Chunks produced: {len(_chunks_s)} via {'span' if used_spans else 'window'} path")
assert len(_chunks_s) > 0, "No chunks produced"

# Assertion B: synthetic contract with no spans (forces window fallback)
_fake = {"text": " ".join(f"word{i}" for i in range(200)), "spans": []}
_cw   = get_eval_chunks(_fake, window=70, stride=35)
assert len(_cw) > 1,                                 "Expected multiple window chunks"
assert len(_cw[0]["text"].split()) <= 70,            "First chunk too large"
assert _cw[1]["text"].split()[0] in _cw[0]["text"], "No overlap between chunks"
print(f"Window fallback OK \u2014 {len(_cw)} chunks from 200-word text")

Contract has top-level spans: True
Chunks produced: 97 via span path
Window fallback OK — 5 chunks from 200-word text


## Section 5 — Anchor Retrieval

Embeds all eval chunks and ranks by cosine similarity to the hypothesis embedding. Returns the top-k anchors with embeddings attached (used as query vectors in Section 6).

In [42]:
# Cell 5 — Anchor retrieval
from numpy.linalg import norm


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (norm(a) * norm(b) + 1e-9))


def get_anchors(
    eval_chunks: list,
    hypothesis_embedding: np.ndarray,
    top_k: int = ANCHOR_TOP_K,
) -> list:
    texts      = [c["text"] for c in eval_chunks]
    embeddings = embedder.encode(texts, batch_size=32, show_progress_bar=False)
    scored = [
        {**chunk, "embedding": emb, "score": cosine_similarity(emb, hypothesis_embedding)}
        for chunk, emb in zip(eval_chunks, embeddings)
    ]
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


# Assertion
_sample_chunks = get_eval_chunks(EVAL_CONTRACTS[0])
_h             = HYPOTHESES["H04"]
_anchors       = get_anchors(_sample_chunks, _h["embedding"], top_k=3)

assert len(_anchors) == 3,                           "Expected 3 anchors"
assert _anchors[0]["score"] >= _anchors[1]["score"], "Anchors not sorted by score"
assert "embedding" in _anchors[0],                   "Anchor missing embedding"
print(f"Anchor retrieval OK \u2014 top score: {_anchors[0]['score']:.3f}")

Anchor retrieval OK — top score: 0.794


## Section 6 — GraphRAG Retrieval (the core)

Over-fetches from the vector index, then filters by graph structure (hypothesis + label).  
`candidate_limit = per_label * 20` gives the graph filter enough candidates to work with.

In [ ]:
# Cell 6 — GraphRAG retrieval
# Graph-filter first (hypothesis + label), then cosine rank within that set.
# min_score threshold drops weak matches before they reach the prompt.

PRECEDENT_QUERY = """
MATCH (a:Annotation)-[:GROUNDS_HYPOTHESIS]->(h:Hypothesis {h_id: $h_id})
WHERE a.label = $label
MATCH (a)-[:SUPPORTED_BY]->(cl:Clause)
WITH cl, vector.similarity.cosine(cl.embedding, $embedding) AS score
WHERE score >= $min_score
ORDER BY score DESC
LIMIT $per_label
RETURN cl.clause_id AS clause_id,
       cl.text      AS text,
       score
"""

MIN_SCORE = 0.7  # lower = more results, higher = stricter — tune in Section 0


def _query_single_anchor(anchor_embedding, h_id: str, per_label: int,
                         min_score: float) -> dict:
    emb_list = anchor_embedding.tolist()
    rows = {"ENTAILED": [], "CONTRADICTED": []}
    with driver.session() as s:
        for label in ("ENTAILED", "CONTRADICTED"):
            rows[label] = s.run(PRECEDENT_QUERY, {
                "embedding": emb_list,
                "h_id":      h_id,
                "label":     label,
                "per_label": per_label,
                "min_score": min_score,
            }).data()
    return rows


def get_precedents(anchors: list, h_id: str, per_label: int = PER_LABEL_K,
                   min_score: float = MIN_SCORE) -> dict:
    seen      = {"ENTAILED": set(), "CONTRADICTED": set()}
    collected = {"ENTAILED": [], "CONTRADICTED": []}

    for anchor in anchors:
        rows = _query_single_anchor(anchor["embedding"], h_id, per_label, min_score)
        for label in ("ENTAILED", "CONTRADICTED"):
            for row in rows[label]:
                if row["clause_id"] not in seen[label]:
                    seen[label].add(row["clause_id"])
                    collected[label].append(row)

    for label in collected:
        collected[label] = sorted(
            collected[label], key=lambda x: x["score"], reverse=True
        )[:per_label]

    return collected


# Assertion
_prec = get_precedents(_anchors, "H04", per_label=3, min_score=MIN_SCORE)
assert "ENTAILED"     in _prec
assert "CONTRADICTED" in _prec
print(f"Precedents OK — ENTAILED:{len(_prec['ENTAILED'])}, CONTRADICTED:{len(_prec['CONTRADICTED'])}")
print(f"(min_score={MIN_SCORE} — lower this value if you get 0 results)")

## Section 7 — Prompt Builder

Builds the dynamic few-shot prompt. Number of examples scales with how many precedents were retrieved — zero retrieved = zero-shot fallback automatically.

In [ ]:
# Cell 7a — Dynamic few-shot prompt builder

def build_prompt(
    h_id: str,
    anchors: list,
    precedents: dict,
    max_few_shots: int = MAX_FEW_SHOTS,
    h_data: dict = None,       # pass explicitly to avoid needing HYPOTHESES global
) -> str:
    # h_data can come from the in-memory HYPOTHESES dict or fetched directly from Neo4j
    h = h_data if h_data is not None else HYPOTHESES[h_id]

    few_shot_lines = []
    for label, symbol in [("ENTAILED", "ENTAILED"), ("CONTRADICTED", "CONTRADICTED")]:
        for p in precedents[label][:max_few_shots]:
            few_shot_lines.append(
                f"[Example — {symbol}]\n"
                f"Clause: {p['text'].strip()}\n"
                f"Verdict: {symbol}"
            )

    few_shot_block = (
        "### Precedents from similar NDAs\n" + "\n\n".join(few_shot_lines) + "\n"
        if few_shot_lines else ""
    )

    evidence_block = "\n".join(f"  • {a['text'].strip()}" for a in anchors)

    return (
        f"You are a legal NDA analyst. Classify one hypothesis based only on the contract evidence provided.\n\n"
        f"## Hypothesis [{h_id}]: {h['title']}\n"
        f"{h['definition']}\n\n"
        f"{few_shot_block}"
        f"### Evidence from the contract under review\n"
        f"{evidence_block}\n\n"
        f"Classify the hypothesis as exactly one of: ENTAILED / CONTRADICTED / NOT_MENTIONED\n"
        f"Then provide a one-sentence justification citing specific evidence.\n\n"
        f"Verdict: "
    )


# Assertions (uses HYPOTHESES from cell_3a — only needed during original setup run)
_prompt    = build_prompt("H04", _anchors, _prec)
n_examples = _prompt.count("[Example")

assert "H04" in _prompt,                        "Missing hypothesis ID"
assert "Evidence from the contract" in _prompt, "Missing evidence section"
assert "Verdict: " in _prompt,                  "Missing verdict prompt"

print(f"Prompt built — {len(_prompt)} chars, {n_examples} few-shot example(s)")
print("\n--- Prompt preview (first 600 chars) ---")
print(_prompt[:600])

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [24]:
from dotenv import load_dotenv

from scripts.graphrag_utils import (
    get_driver,
    get_eval_chunks,
    get_anchors,
    get_precedents,
    build_prompt,
    fetch_hypothesis,
)

# Load credentials from /content/.env
load_dotenv("/content/.env", override=True)

NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USER     = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver   = get_driver(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embedder = SentenceTransformer(EMBED_MODEL)
print(f"Embedding model loaded: {EMBED_MODEL}")
# --- Edit these ---
EVAL_CONTRACT_IDX = 0       # 0 to len(EVAL_CONTRACTS)-1
HYPOTHESIS_ID     = "H04"   # H01 to H17
ANCHOR_TOP_K_RUN  = 3
PER_LABEL_K_RUN   = 5
MAX_FEW_SHOTS_RUN = 5

# ── Pipeline ──────────────────────────────────────────────────────────────────
contract   = EVAL_CONTRACTS[EVAL_CONTRACT_IDX]
h_data     = fetch_hypothesis(HYPOTHESIS_ID, driver)

chunks     = get_eval_chunks(contract)
anchors    = get_anchors(chunks, h_data["embedding"], embedder, top_k=ANCHOR_TOP_K_RUN)
precedents = get_precedents(anchors, HYPOTHESIS_ID, driver, per_label=PER_LABEL_K_RUN)
prompt     = build_prompt(HYPOTHESIS_ID, anchors, precedents, h_data,
                          max_few_shots=MAX_FEW_SHOTS_RUN)

print(f"Contract  : {contract['id']}")
print(f"Hypothesis: {HYPOTHESIS_ID} — {h_data['title']}")
print(f"Chunks    : {len(chunks)}  |  Anchors: {len(anchors)}")
print(f"Precedents: ENTAILED={len(precedents['ENTAILED'])}  CONTRADICTED={len(precedents['CONTRADICTED'])}")
print(f"Prompt    : {len(prompt)} chars, {prompt.count('[Example')} few-shot example(s)")
print("\n--- Full prompt ---")
print(prompt)

ImportError: cannot import name 'get_driver' from 'scripts.graphrag_utils' (/content/scripts/graphrag_utils.py)